In [2]:
from pyspark.sql import functions as F

df_silver_product = (
    spark.read
        .format("delta")
        .load("abfss://Dell_SEO_Analytics@onelake.dfs.fabric.microsoft.com/LH_SEO_Silver.Lakehouse/Tables/dbo/dim_product")
)

print("Silver dim_product rows:", df_silver_product.count())
print("Silver dim_product columns:", len(df_silver_product.columns))

display(df_silver_product)

StatementMeta(, 6540b6be-338b-47b0-82da-9c8fb5d91a6e, 3, Finished, Available, Finished, False)

Silver dim_product rows: 50
Silver dim_product columns: 7


SynapseWidget(Synapse.DataFrame, 67b5035e-e685-41c3-8bcf-8229789f557d)

In [3]:
from pyspark.sql.window import Window

window_product = Window.orderBy("product_id")

df_gold_product = (
    df_silver_product
    .withColumn(
        "product_key",
        F.row_number().over(window_product)
    )
)

display(
    df_gold_product
    .orderBy("product_key")
)

StatementMeta(, 6540b6be-338b-47b0-82da-9c8fb5d91a6e, 4, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, b0baecce-a215-4928-8b04-3cc36faa89e0)

In [4]:
(
    df_gold_product.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("dim_product")
)

StatementMeta(, 6540b6be-338b-47b0-82da-9c8fb5d91a6e, 5, Finished, Available, Finished, False)

In [5]:
df_gold_product_check = spark.read.table("dim_product")

print("Gold dim_product rows:", df_gold_product_check.count())
print("Gold dim_product columns:", len(df_gold_product_check.columns))

display(df_gold_product_check.orderBy("product_key"))

StatementMeta(, 6540b6be-338b-47b0-82da-9c8fb5d91a6e, 6, Finished, Available, Finished, False)

Gold dim_product rows: 50
Gold dim_product columns: 8


SynapseWidget(Synapse.DataFrame, 4850aa61-fdae-4931-b292-349e3fe22ef7)